# Session 33: Classification with Logistic Regression

Data Science with Python, Level 2, Stage 5: ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Tell classification from regression.
2. Explain the sigmoid function.
3. Fit a logistic regression with scikit-learn.
4. Use predicted probabilities and a threshold.
5. Compare accuracy with a baseline.
6. Interpret coefficients as changes in odds.

## Where We Are

In Sessions 31 and 32 the target was a number: a price. Many business questions have a **category** as the answer instead: will this customer leave or stay? Is this transaction fraud? Will this student pass? Predicting a category is called **classification**.

The most common starting model is **logistic regression**. Despite its name, it is a classification model: it predicts the **probability** of the yes answer. This session uses the 2,000 practice phone-company customers from Session 27; about 30 percent of them left.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)

## 1. From Numbers to Categories

In [ ]:
sns.stripplot(data=churn, x="tenure_months", y="churn",
              alpha=0.2, size=3)
plt.title("Leavers had shorter tenure")
plt.xlabel("Months as a customer")
plt.show()

## 2. The Sigmoid Function

Logistic regression first computes a weighted sum, exactly like linear regression: `z = w * x + b`. Then it passes `z` through the **sigmoid** function, `1 / (1 + e^(-z))`, which turns any number into a value between 0 and 1. Large positive `z` gives a probability close to 1; large negative `z` close to 0.

In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.plot(z, sigmoid)
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("The sigmoid turns any number into a probability")
plt.xlabel("z = w * x + b")
plt.ylabel("Probability")
plt.show()

## 3. Fitting a Logistic Regression

In [ ]:
model = LogisticRegression()
model.fit(churn[["tenure_months"]], churn["left"])

new = pd.DataFrame({"tenure_months": [1, 12, 36, 60]})
new["P(leave)"] = model.predict_proba(new)[:, 1].round(3)
print(new)

In [ ]:
months = pd.DataFrame({"tenure_months": range(73)})
risk = model.predict_proba(months)[:, 1]
plt.plot(months, risk, color="tab:red")
plt.title("Predicted risk by tenure")
plt.xlabel("Months as a customer")
plt.ylabel("P(leave)")
plt.show()

## 4. Many Features and a Fair Test

In [ ]:
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)

features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X = churn[features].astype(float)
y = churn["left"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler().fit(X_train)
model = LogisticRegression()
model.fit(scaler.transform(X_train), y_train)

pred = model.predict(scaler.transform(X_test))
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Baseline, always 'stays':", round(1 - y_test.mean(), 3))

## 5. Thresholds and the Confusion Matrix

In [ ]:
print(confusion_matrix(y_test, pred))

In [ ]:
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

for t in [0.3, 0.5, 0.7]:
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    print(f"threshold {t}: caught {tp} of {tp + fn} leavers, {fp} false alarms")

## 6. Reading the Coefficients

In [ ]:
odds = pd.Series(np.exp(model.coef_[0]), index=features).sort_values()
print(odds.round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Fooled by accuracy | Reporting accuracy without a baseline | Compare with always predicting the most common class. |
| Leakage | Fitting the scaler on all the data before splitting | Fit the scaler on the training set only. |
| Wrong column | Using `predict_proba(X)` as one column | Take `[:, 1]` for the probability of the positive class. |
| Fixed threshold | Always using 0.5 | Choose the threshold from the costs of each kind of error. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: One-Feature Models

Fit a logistic regression for `left` on each of `monthly_charges` and `support_calls`. For each, print the probability of leaving for a low and a high value.

In [ ]:
# Write your code below

### In-Class Exercise 2: Risky Customers

Using the model with all features from the lecture, list the ten test customers with the highest predicted risk, with their contract, tenure and whether they actually left.

In [ ]:
# Write your code below

### In-Class Exercise 3: Exam Pass Model

Load `exam_scores.csv` and predict `passed` from `study_hours_week`, `sleep_hours` and `attendance_pct`. Report the test accuracy and the baseline.

In [ ]:
# Write your code below

### In-Class Exercise 4: Choose a Threshold

Suppose a missed leaver costs 300 and a wasted offer costs 40. For thresholds from 0.1 to 0.9 in steps of 0.1, compute the total cost on the test set. Which threshold is cheapest?

In [ ]:
# Write your code below

## Summary

- Classification predicts a category; regression a number.
- The sigmoid turns a weighted sum into a probability.
- `predict_proba(X)[:, 1]` gives the probability of the yes class.
- Always compare accuracy with the majority-class baseline.
- The confusion matrix shows what kind of mistakes a model makes.
- The threshold is a business choice; odds ratios explain features.

## What's Next

Accuracy hid that the churn model missed most leavers. Next session you will measure models properly: precision, recall, F1 and ROC curves for classifiers, and MAE, RMSE and R2 for regression.